# 06 — Social chart (Pillow)

The publication-ready lead chart, using the shared `lollipop` template in
**dumbbell mode**: each film shows its inflation-adjusted gross (teal) vs its
original nominal gross (gold); the connector length is the inflation gap.

The chart renders **inline** and is saved to `outputs/social/`.

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

# make the workspace-level shared/ importable
WORKSPACE = PROJECT.parent.parent
sys.path.insert(0, str(WORKSPACE / 'shared'))

from src.ingest import load_config
from src.clean_quality import get_connection
from chart_templates import lollipop
from viz import PRESETS
from IPython.display import display

cfg = load_config('config.yaml')
con = get_connection(cfg)
films = con.execute('''
    SELECT title, adjusted_gross, nominal_gross, release_year
    FROM films_adjusted ORDER BY adjusted_gross DESC LIMIT 15
''').df()
con.close()
films['label'] = films['title'] + '  (' + films['release_year'].astype(str) + ')'
films.head()

## Render the dumbbell lead chart

In [ ]:
def money(v):
    return f'${v/1e9:.2f}B' if v >= 1e9 else f'${v/1e6:.0f}M'

img_w, img_h, _ = PRESETS['twitter_landscape']
img = lollipop(
    films, category_col='label',
    value_col='adjusted_gross', value2_col='nominal_gross',
    value_fmt=money,
    title="Hollywood's real box-office champions, once you adjust for inflation",
    subtitle='Top 15 domestic films by ticket-price-adjusted gross (2022 $) vs. their original nominal gross',
    source='Box Office Mojo, Top Lifetime Adjusted Grosses (domestic, adj. to 2022)',
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='Adjusted (2022 $)', value2_label='Nominal (release $)',
    img_width=img_w, img_height=img_h,
)

# Workspace rule: ALWAYS show the chart inline, then save. display() first
# so a chart can never be saved without being rendered in the notebook.
display(img)

out = Path(cfg['paths']['outputs_social']); out.mkdir(parents=True, exist_ok=True)
path = out / '01_adjusted_vs_nominal_top_films.png'
img.save(path)
print('Displayed above; also saved ->', path)

---
Chart written to `outputs/social/`. Watermark `@unwelcomedata`, brand palette,
`twitter_landscape` preset.